### 1. Import Libraries

In [ ]:
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

try:
    import mlxtend
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlxtend"])

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

### 2. Load Dataset


In [15]:
orders = pd.read_csv("../Data/orders.csv")
order_items = pd.read_csv("../Data/order_items.csv")
products = pd.read_csv("../Data/products.csv")

print("Orders shape:", orders.shape)
print("Order items shape:", order_items.shape)
print("Products shape:", products.shape)

Orders shape: (33580, 10)
Order items shape: (59163, 5)
Products shape: (1197, 6)


### 3. Preview the Data

In [16]:
orders.head()

,order_id,customer_id,order_time,payment_method,discount_pct,subtotal_usd,total_usd,country,device,source
0,1,13917,2025-01-31T23:07:42,card,20,107.15,85.72,PL,desktop,organic
1,2,1022,2024-02-19T01:17:50,card,0,116.17,116.17,FR,tablet,organic
2,3,6145,2024-12-04T20:24:13,card,0,137.35,137.35,US,mobile,organic
3,4,3152,2024-07-17T08:50:47,card,15,32.18,27.35,BR,mobile,email
4,5,12378,2020-08-21T16:54:16,card,0,238.09,238.09,NL,desktop,paid


In [17]:
order_items.head()

,order_id,product_id,unit_price_usd,quantity,line_total_usd
0,1,226,107.15,1,107.15
1,2,771,116.17,1,116.17
2,3,415,94.49,1,94.49
3,3,24,42.86,1,42.86
4,4,1157,32.18,1,32.18


In [18]:
products.head()

,product_id,category,name,price_usd,cost_usd,margin_usd
0,1,Electronics,SSD MediumBlue 149,570.28,352.69,217.59
1,2,Electronics,Keyboard DeepPink 696,498.13,263.13,235.00
2,3,Electronics,Headphones Orchid 188,548.53,309.60,238.93
3,4,Electronics,Smartwatch BurlyWood 664,268.36,153.56,114.80
4,5,Electronics,Smartwatch Cornsilk 328,63.69,42.65,21.04


### 4. Select Needed Columns and Merge Data


In [19]:
merged = order_items.merge(
    products[["product_id", "name"]],
    on="product_id",
    how="inner"
)

merged.head()

,order_id,product_id,unit_price_usd,quantity,line_total_usd,name
0,1,226,107.15,1,107.15,Toaster MediumSlateBlue 575
1,2,771,116.17,1,116.17,Socks Orange 300
2,3,415,94.49,1,94.49,Shampoo LawnGreen 601
3,3,24,42.86,1,42.86,SSD Orchid 272
4,4,1157,32.18,1,32.18,Board Game DarkSeaGreen 297


### 5. Build Transactions


In [20]:
transactions = merged.groupby("order_id")["name"].apply(list).tolist()

print("Number of transactions:", len(transactions))
transactions[:5]

Number of transactions: 33580


[['Toaster MediumSlateBlue 575'],
 ['Socks Orange 300'],
 ['Shampoo LawnGreen 601', 'SSD Orchid 272'],
 ['Board Game DarkSeaGreen 297'],
 ['Socks GhostWhite 821', 'Jacket RosyBrown 185']]

### 6. Clean Transactions


In [26]:
clean_transactions = []

for transaction in transactions:
    clean_transaction = []

    for item in transaction:
        item = str(item).strip()

        if item != "" and item not in clean_transaction:
            clean_transaction.append(item)

    if len(clean_transaction) > 0:
        clean_transactions.append(clean_transaction)

print("Number of clean transactions:", len(clean_transactions))
clean_transactions[:5]


Number of clean transactions: 33580


[['Toaster MediumSlateBlue 575'],
 ['Socks Orange 300'],
 ['Shampoo LawnGreen 601', 'SSD Orchid 272'],
 ['Board Game DarkSeaGreen 297'],
 ['Socks GhostWhite 821', 'Jacket RosyBrown 185']]

### 7. One-Hot Encoding


In [33]:
te = TransactionEncoder()
te_data = te.fit(clean_transactions).transform(clean_transactions)

basket = pd.DataFrame(te_data, columns=te.columns_)

print("Basket shape:", basket.shape)
basket.head()
basket.to_csv("../Data/Basket.csv")

Basket shape: (33580, 1195)
